# Task 2 – Data Engineering

This notebook performs data quality checks, cleaning, feature engineering, and builds a full PySpark preprocessing pipeline.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
import os, glob, time

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Load data (same path as Task1)
data_path = os.path.join(os.getcwd(), 'data', 'yellow_tripdata_2025-*.parquet')
df = spark.read.parquet(data_path)
df = df.cache()

## Missing Value Analysis

In [ ]:
missing_counts = df.select([F.count(F.when(F.isnan(c) | F.isnull(c), c)).alias(c) for c in df.columns])
missing_counts.show(truncate=False)  # <--- Screenshot this table

## Duplicate Record Analysis

In [ ]:
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_rows = total_rows - distinct_rows
print(f'Duplicate rows: {duplicate_rows:,}')  # <--- Screenshot this output

## Data Type Summary

In [ ]:
for name, dtype in df.dtypes:
    print(f'{name}: {dtype}')

## Data Cleaning
* Remove rows with null fare_amount
* Filter unrealistic fares (0 < fare <= 500)
* Filter nonsensical coordinates if needed

In [ ]:
clean_df = df.dropna(subset=['fare_amount'])
clean_df = clean_df.filter((F.col('fare_amount') > 0) & (F.col('fare_amount') < 500))
# Additional geographic filters (optional)
clean_df = clean_df.filter(F.col('pickup_longitude').between(-74.5, -73.5) &
                                 F.col('pickup_latitude').between(40.5, 41.5) &
                                 F.col('dropoff_longitude').between(-74.5, -73.5) &
                                 F.col('dropoff_latitude').between(40.5, 41.5))

## Feature Engineering
Create temporal features and trip duration.

In [ ]:
fe_df = clean_df
    .withColumn('pickup_datetime', F.to_timestamp('tpep_pickup_datetime'))
    .withColumn('dropoff_datetime', F.to_timestamp('tpep_dropoff_datetime'))
    .withColumn('pickup_hour', F.hour('pickup_datetime'))
    .withColumn('pickup_day', F.dayofmonth('pickup_datetime'))
    .withColumn('pickup_month', F.month('pickup_datetime'))
    .withColumn('trip_duration', (F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime')).cast('double'))

## Encoding and Scaling – PySpark Pipeline

In [ ]:
# Categorical columns to index
cat_cols = ['vendor_id', 'store_and_fwd_flag']
indexers = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in cat_cols]
# One‑hot encode indexed columns
encoders = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_vec') for c in cat_cols]
# Numerical columns to assemble
num_cols = ['passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude',
                    'dropoff_longitude', 'dropoff_latitude', 'trip_duration',
                    'pickup_hour', 'pickup_day', 'pickup_month']
assembler = VectorAssembler(inputCols=num_cols + [c+'_vec' for c in cat_cols], outputCol='features_assembled')
scaler = StandardScaler(inputCol='features_assembled', outputCol='features', withMean=True, withStd=True)
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
pipeline_model = pipeline.fit(fe_df)
prepared_df = pipeline_model.transform(fe_df)
# Show partition count – screenshot ready
print(f'Partitions: {prepared_df.rdd.getNumPartitions()}')  # <--- Screenshot this output

## Pipeline Stages Explanation
1. **StringIndexer** – converts categorical strings to integer indices.
2. **OneHotEncoder** – creates binary vectors for each category, avoiding ordinal bias.
3. **VectorAssembler** – merges numeric columns and one‑hot vectors into a single feature vector.
4. **StandardScaler** – standardises each feature to zero mean and unit variance, improving convergence for many models.